# Batch Gradient Descent Class with Early Stopping

In [197]:
import numpy as np

class BatchGradientDescent:
    def __init__(self, num_epochs, learning_rate):
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate

    def fit(self, X_train, y_train, X_val, y_val):

        y_train = y_train.to_numpy()
        y_val = y_val.to_numpy()
        
        best_theta = None
        best_valid_error = float('inf')
        theta = np.ones((3, X_train.shape[1]))  # Initialize theta to ones
        
        for epoch in range(self.num_epochs):
        
            scores = X_train @ theta.T
            
            for k in range(3):
                gradient_vector = np.zeros(X_train.shape[1])  # Initialize gradient vector to zeros
                for i in range(X_train.shape[0]):
                    denominator_probability = 0
                    for j in range(3):
                        denominator_probability += np.exp(scores[i, j])
                    probability = np.exp(scores[i, k]) / denominator_probability
                    gradient_vector += X_train[i] * (probability - (y_train[i] == k))
                gradient_vector /= X_train.shape[0]
                theta[k] -= self.learning_rate * gradient_vector

            # Compute the validation error
            val_scores = X_val @ theta.T
            val_pred = np.argmax(val_scores, axis=1)
            val_error = sum(val_pred != y_val) / len(y_val)

            if val_error < best_valid_error:
                best_valid_error = val_error
                print(f"Epoch {epoch + 1}: Validation error improved to {best_valid_error:.4f}")
                best_theta = theta.copy()

        return best_theta

# Load the data

In [198]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

# Split the data

In [199]:
X = iris.data[["petal length (cm)", "petal width (cm)"]]
y = iris.target

test_ratio = 0.2
validation_ratio = 0.2

total_size = len(X)
test_size = int(total_size * test_ratio)
validation_size = int(total_size * validation_ratio)
train_size = total_size - test_size - validation_size

rng = np.random.default_rng(seed=42)
rnd_indices = rng.permutation(total_size)
X_train = X.iloc[rnd_indices[:train_size]]
y_train = y.iloc[rnd_indices[:train_size]]
X_val = X.iloc[rnd_indices[train_size:train_size+validation_size]]
y_val = y.iloc[rnd_indices[train_size:train_size+validation_size]]
X_test = X.iloc[rnd_indices[train_size+validation_size:]]
y_test = y.iloc[rnd_indices[train_size+validation_size:]]

# Standardize the data

In [200]:
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)
X_train_scaled = (X_train - mean) / (std + 1e-8)
X_val_scaled = (X_val - mean) / (std + 1e-8)
X_test_scaled = (X_test - mean) / (std + 1e-8)


# Add Bias

In [201]:
X_train_scaled_with_bias = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_val_scaled_with_bias = np.c_[np.ones((X_val_scaled.shape[0], 1)), X_val_scaled]
X_test_scaled_with_bias = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

# Create and train the model

In [202]:
model = BatchGradientDescent(num_epochs=10000, learning_rate=0.01)
best_theta = model.fit(X_train_scaled_with_bias, y_train, X_val_scaled_with_bias, y_val)
best_theta

Epoch 1: Validation error improved to 0.1667
Epoch 163: Validation error improved to 0.1333
Epoch 288: Validation error improved to 0.1000
Epoch 292: Validation error improved to 0.0667
Epoch 436: Validation error improved to 0.0333


array([[0.80971218, 0.23858127, 0.27249337],
       [1.33312176, 1.16598338, 1.07965626],
       [0.85716606, 1.59543534, 1.64785037]])

# Prediction on the test dataset

In [203]:
X_test_scores = X_test_scaled_with_bias @ best_theta.T
X_test_pred = np.argmax(X_test_scores, axis=1)

X_test_error = sum(X_test_pred != y_test) / len(y_test)
print(f"Overall accuracy on the test dataset: {1-X_test_error:.4f}")

Overall accuracy on the test dataset: 0.9667
